# Phase 4: SVM Integration and Analysis
**CSC14120 - Parallel Programming**

## Objectives
- Load trained autoencoder weights from Phase 3
- Extract 8192-dimensional features using encoder
- Train SVM classifier with LIBSVM (RBF kernel, C=10, gamma=auto)
- Evaluate on test set and generate confusion matrix
- Target accuracy: 60-65%

## Pipeline
1. **Feature Extraction**: Encoder forward pass on all 60K images
2. **SVM Training**: Train on 50K training features with labels
3. **Evaluation**: Predict on 10K test features, compute accuracy

## Huong dan:
1. Zip project (khong bao gom data/)
2. Upload len Colab
3. Upload file `phase3_opt.weights` (hoac de trong project zip)
4. Chay tat ca cells

In [ ]:
# Check GPU and CUDA
!nvidia-smi
!nvcc --version

In [ ]:
# Upload and extract project
from google.colab import files
import zipfile
import os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        os.chdir(root)
        break

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10 dataset
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

In [ ]:
# Install LIBSVM
print("Installing LIBSVM...")
!apt-get update -qq
!apt-get install -y -qq libsvm-dev libsvm-tools

# Clone and build LIBSVM for C++ integration
!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src
!cd libsvm_src && make lib
!cp libsvm_src/svm.h include/
!cp libsvm_src/svm.cpp src/

print("LIBSVM installed!")

In [ ]:
# Upload Phase 3 weights if not in project
weights_file = 'phase3_opt.weights'

if not os.path.exists(weights_file):
    print(f"Weights file '{weights_file}' not found. Please upload:")
    uploaded = files.upload()
    if uploaded:
        weights_file = list(uploaded.keys())[0]
        print(f"Using uploaded weights: {weights_file}")
else:
    print(f"Found weights file: {weights_file}")

!ls -lh {weights_file}

In [ ]:
# Build Phase 4 executable with LIBSVM support
# -DWITH_SVM: Enable SVM integration
# -DWITH_LIBSVM: Use real LIBSVM instead of stub
# -lcudnn: cuDNN for optimized convolution

!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o phase4_svm \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

print('Phase 4 build complete!')

In [ ]:
# Run Phase 4: Feature Extraction + SVM Training
# Using all 50000 training samples, C=10, gamma=auto
!./phase4_svm --data data --weights phase3_opt.weights \
    --log phase4_svm.txt --csv phase4_results.csv \
    --svm-c 10.0

In [ ]:
# Visualize results
import pandas as pd
import matplotlib.pyplot as plt

# Load results
results = pd.read_csv('phase4_results.csv')
results_dict = dict(zip(results['metric'], results['value']))

print("=" * 60)
print("PHASE 4 RESULTS SUMMARY")
print("=" * 60)
print(f"Test Accuracy: {results_dict['test_accuracy']*100:.2f}%")
print(f"Training Samples: {int(results_dict['train_samples'])}")
print(f"Test Samples: {int(results_dict['test_samples'])}")
print(f"Feature Dimension: {int(results_dict['feature_dim'])}")
print(f"SVM C: {results_dict['svm_c']}")
print(f"SVM gamma: {results_dict['svm_gamma']:.6e}")
print("=" * 60)
print("TIMING:")
print(f"  Feature Extraction: {results_dict['feature_extraction_time']:.2f}s")
print(f"  SVM Training: {results_dict['svm_training_time']:.2f}s")
print(f"  Evaluation: {results_dict['evaluation_time']:.2f}s")
print(f"  Total: {results_dict['total_time']:.2f}s")
print("=" * 60)

In [ ]:
# Create timing breakdown chart
times = {
    'Feature\nExtraction': results_dict['feature_extraction_time'],
    'SVM\nTraining': results_dict['svm_training_time'],
    'Evaluation': results_dict['evaluation_time']
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for timing
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax1.bar(times.keys(), times.values(), color=colors)
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Phase 4 Timing Breakdown')
ax1.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, times.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}s', ha='center', va='bottom', fontsize=10)

# Pie chart for time distribution
ax2.pie(times.values(), labels=times.keys(), autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('Time Distribution')

plt.tight_layout()
plt.savefig('phase4_timing.png', dpi=150)
plt.show()

In [ ]:
# Show detailed log
print("=" * 60)
print("DETAILED LOG")
print("=" * 60)
!cat phase4_svm.txt

In [ ]:
# Download results
files.download('phase4_svm.txt')
files.download('phase4_results.csv')
files.download('phase4_timing.png')

# Download SVM model if exists
if os.path.exists('phase4_svm.model'):
    files.download('phase4_svm.model')

## Performance Analysis

### Phase 4 Pipeline

| Step | Description | Target Time |
|:-----|:------------|:------------|
| Feature Extraction | Encoder forward pass on 60K images | < 20 seconds |
| SVM Training | LIBSVM with RBF kernel | Varies by samples |
| Evaluation | Predict on 10K test images | < 5 seconds |

### SVM Configuration (per project spec)

| Parameter | Value | Description |
|:----------|:------|:------------|
| Kernel | RBF | Radial Basis Function |
| C | 10 | Regularization parameter |
| gamma | auto (1/dim) | Kernel coefficient |

### Expected Results

- **Target Accuracy**: 60-65%
- **Feature Dimension**: 8192 (8x8x128 latent representation)